Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-gemma/gbench/blob/main/examples/notebooks/04_academic_evaluations_and_reasoning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench academic evaluations, reasoning mode, and multimodal budgets

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook explores how to measure the reasoning accuracy of foundation models using `gbench --evals`. You will run native Python evaluation harnesses for science, mathematics, and multimodal understanding, configuring Chain-of-Thought reasoning (`--eval-thinking`) and vision soft-token budgets.

You can point `gbench` at **any OpenAI-compatible `/v1` endpoint** via `--remote-endpoint`. This notebook drives a **local Ollama server** serving a quantized Gemma 4 GGUF, and also shows the **vLLM remote-endpoint** path (`--remote-endpoint http://127.0.0.1:8000/v1 --tokenizer google/gemma-4-26B-A4B-it`, served model id `google/gemma-4-26B-A4B-it`) alongside it.

> **Note:** the Gemma 4 GGUF (`unsloth/gemma-4-E4B-it-qat-GGUF`) and tokenizer/model names (`google/gemma-4-E4B-it`, `google/gemma-4-26B-A4B-it`) are the intended Gemma 4 launch artifacts and are placeholders until the model is public.

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Execute native Python academic evaluation suites (`gpqa_diamond`, `mmlu_pro`, `mmmu_pro`) against an OpenAI-compatible endpoint.
3. Enable Chain-of-Thought reasoning mode (`--eval-thinking`) on thinking-aware suites and analyze its impact on reasoning accuracy.
4. Provide the required `--max-output-tokens` budget (65536 with thinking on) so long reasoning chains are not truncated.
5. Load the per-suite `eval_<suite>_<model>_<format>.json` result files to inspect scores.
6. Run a multimodal vision suite (`mmmu_pro`) and understand soft-token budgeting (server-side when using `--remote-endpoint`).
7. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-gemma/gbench)
* [GPQA science evaluation benchmark](https://github.com/idavidrein/gpqa)
* [MMLU-Pro reasoning benchmark](https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro)
* [MMMU-Pro multimodal benchmark](https://mmmu-benchmark.github.io/)

## 1. Environment setup and installation

We clone the `gbench` repository from GitHub, change directory into the project root (`%cd gbench`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-gemma/gbench.git
    %cd -q /content/gbench
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("gbench").is_dir():
            !git clone https://github.com/google-gemma/gbench.git
        %cd gbench

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available evaluation benchmark suites and pillars
!gbench --list evals
!gbench --list pillars

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus tokenizers and eval datasets (e.g. GPQA) from the Hugging Face Hub with `huggingface_hub` — some are **gated** — so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model/dataset page you use (e.g. GPQA).
2. On **Colab**: click the **🔑 key icon (Secrets)** in the left sidebar → **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector),
# tokenizers, and eval datasets (e.g. GPQA) from the Hugging Face Hub via
# huggingface_hub, which authenticates with it (resumable, higher rate limits, and
# access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless BOTH the binary and its llama-server runner are present.
# A binary-only partial install fails every request with "llama-server binary not
# found", so checking only for the binary would skip the repair.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

We download here rather than letting `ollama create` pull `hf.co/…` itself: Ollama's puller is anonymous (it can't use `HF_TOKEN`) and can stall on the HF CDN. The two-`FROM` import (main GGUF + `mmproj`) keeps the model's **vision** capability. If you prefer vLLM over Ollama, skip this section and serve with `vllm serve google/gemma-4-26B-A4B-it --port 8000`, then pass `--remote-endpoint http://127.0.0.1:8000/v1 --tokenizer google/gemma-4-26B-A4B-it` to `gbench`.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) via huggingface_hub, which
# authenticates with HF_TOKEN - Ollama's own hf.co puller is anonymous and can stall
# on the HF CDN. Build the model FROM the local files: a two-FROM import (main +
# mmproj) keeps gemma-4's vision capability (verified with `ollama show`). realpath
# resolves the HF cache symlink so `ollama create` reads the actual file.
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 8192',
          'SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk — no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Running academic benchmarks

We can inspect all registered evaluation suites and capability pillars via `!gbench --list evals` and `!gbench --list pillars`.

Then we run `gbench --evals-only` on `gpqa_diamond` (graduate-level science) and `mmlu_pro` (multi-domain reasoning). To keep the notebook runnable on a laptop/Colab box, the run below is **no-think** and uses `--eval-limit 20` (20 questions **per suite**) — a subset gbench flags as non-leaderboard-comparable. `--max-output-tokens` is **required** for any eval run.

To measure the model's **reasoning** (Chain-of-Thought) instead, use the *optional, heavier* thinking cell below — it is commented out so it never runs by accident.

In [ ]:
# List all registered evaluation benchmark suites
!gbench --list evals

# Academic evals, NO-THINK and limited to a small subset so the notebook runs on
# modest hardware. --eval-limit 20 runs 20 questions PER SUITE; --max-output-tokens
# is REQUIRED for any eval run (65536 is just a cap here - a no-think answer stops
# well before it). gbench flags a subset run as non-leaderboard-comparable.
# (To measure the model's reasoning instead, see the optional thinking cell below.)
!gbench --evals-only \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --evals gpqa_diamond mmlu_pro \
        --max-output-tokens 65536 \
        --eval-limit 20 \
        --results-dir ./results_evals

# --- Alternative: serve with vLLM instead of Ollama ---
# In another shell:  vllm serve google/gemma-4-26B-A4B-it --port 8000
#   !gbench --evals-only \
#           --remote-endpoint http://127.0.0.1:8000/v1 \
#           --tokenizer google/gemma-4-26B-A4B-it \
#           --evals gpqa_diamond mmlu_pro \
#           --max-output-tokens 65536 \
#           --eval-limit 20 \
#           --results-dir ./results_evals

### Optional: reasoning (thinking) mode

The runs in this notebook are **no-think** on a small subset so it stays light on modest hardware. gemma-4 can also reason (Chain-of-Thought) — a *different, heavier* measurement. The next cell shows how; it is **commented out** so nothing heavy runs by accident. Uncomment it (and keep `--eval-limit` small, e.g. 5) to measure reasoning mode.

In [ ]:
# OPTIONAL: reasoning (thinking) mode - HEAVIER, so it is commented out and does NOT
# run by default. It measures the model's Chain-of-Thought (a different, slower thing
# than the no-think run above). Uncomment to run it, and keep --eval-limit small
# (e.g. 5) because reasoning chains are long on modest hardware.
# !gbench --evals-only \
#         --remote-endpoint http://localhost:11434/v1 \
#         --tokenizer google/gemma-4-E4B-it \
#         --evals gpqa_diamond mmlu_pro \
#         --eval-thinking \
#         --max-output-tokens 65536 \
#         --eval-limit 5 \
#         --results-dir ./results_evals_cot

## 8. Loading and inspecting per-suite results

`gbench` writes one JSON file per suite into a timestamped subdirectory of `--results-dir`, named `eval_<suite>_<model>_<format>.json` (e.g. `eval_gpqa_diamond_gemma-4-E4B-it_...json`). The cell below recursively globs those files from `./results_evals` and prints each suite's headline accuracy so you can compare `gpqa_diamond` and `mmlu_pro`.

In [ ]:
import json
from pathlib import Path

results_root = Path("./results_evals")
result_files = sorted(results_root.rglob("eval_*.json"))
if not result_files:
    print(f"No eval_*.json files found under {results_root} - run section 7 first.")
for f in result_files:
    data = json.loads(f.read_text())
    acc = data.get("accuracy")
    # accuracy is already a percentage (0-100), so print it as-is.
    acc_str = f"{acc:.1f}%" if isinstance(acc, (int, float)) else "n/a"
    print(
        f"{f.name}\n"
        f"  status:          {data.get('status', 'ok')}\n"
        f"  thinking:        {data.get('thinking', False)}\n"
        f"  correct/total:   {data.get('correct_answers', '?')}/{data.get('total_questions', '?')}\n"
        f"  accuracy:        {acc_str}\n"
    )

## 9. Running a multimodal vision benchmark

Vision suites such as `mmmu_pro` (and `chartqa`) test image understanding. The run below evaluates `mmmu_pro` through the same endpoint — **no-think** with `--eval-limit 20` to stay light on modest hardware, like the other runs.

**Soft-token budgeting:** the number of vision soft tokens a chart or figure is encoded into matters for accuracy. When you drive a server with `--remote-endpoint`, that budget is fixed **server-side** (Ollama/vLLM), so gbench rejects `--eval-max-soft-tokens` in that mode. The `--eval-max-soft-tokens` flag applies only to vision suites (`mmmu_pro`, `screenspot`, `semantic_keypoint`, `textvqa`, `infographicvqa`, `bundled_detection`) when gbench launches its **own local vLLM** server (no `--remote-endpoint`).

As before, `--max-output-tokens` is required.

In [ ]:
# Multimodal vision eval through the remote endpoint (soft tokens fixed server-side,
# so --eval-max-soft-tokens is NOT passed here). No-think + a small subset
# (--eval-limit 20) so it runs on modest hardware. --max-output-tokens is required.
!gbench --evals-only \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --evals mmmu_pro \
        --max-output-tokens 4096 \
        --eval-limit 20 \
        --results-dir ./results_evals_mm

# --- Local vLLM path (gbench launches its own server) where --eval-max-soft-tokens applies ---
# Omit --remote-endpoint so gbench starts vLLM itself, then you may budget soft tokens:
#   !gbench --evals-only \
#           --evals mmmu_pro \
#           --eval-max-soft-tokens 1120 \
#           --max-output-tokens 4096 \
#           --eval-limit 20 \
#           --results-dir ./results_evals_mm

## 10. Session cleanup and server shutdown

We terminate background Ollama server processes and remove temporary Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")